In [2]:
# Пункт 2.3.1: Импорты, seed и устройство

import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import json
import os
from pathlib import Path

# Фиксируем seed для воспроизводимости
def set_seed(seed=42):
    """Устанавливает seed для всех генераторов случайных чисел"""
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    
    # Для воспроизводимости на CUDA
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    print(f"Seed установлен: {seed}")

# Устанавливаем seed
SEED = 42
set_seed(SEED)

# Определяем устройство - CUDA
# Определяем устройство
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")

# Проверка доступности CUDA
if torch.cuda.is_available():
    print(f"CUDA доступна: {torch.cuda.is_available()}")
    print(f"Количество GPU: {torch.cuda.device_count()}")
    print(f"Имя GPU: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA не доступна, используется CPU")


Seed установлен: 42
Используемое устройство: cpu
CUDA не доступна, используется CPU


In [ ]:
# Пункт 2.3.2: Данные и DataLoader (CIFAR10)

# 1) Загрузка датасета через torchvision
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset_full = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

# 2) Разбиение train/val с фиксированным seed
train_size = int(0.8 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(
    train_dataset_full, 
    [train_size, val_size],
    generator=generator
)

# 3) Создание DataLoader'ов
BATCH_SIZE = 128

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,
    generator=generator
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False
)

# 4) Sanity-check
data_iter = iter(train_loader)
images, labels = next(data_iter)

print(f"x.shape: {images.shape}")  # [128, 3, 32, 32] для CIFAR10
print(f"y.shape: {labels.shape}")
print(f"x range: [{images.min():.3f}, {images.max():.3f}]")
print(f"y unique: {torch.unique(labels)}")

In [ ]:
# Пункт 2.3.3: Модель MLP и цикл обучения (CIFAR10)

# 1) MLP как nn.Module (для CIFAR10 input_size=3072: 3*32*32)
class MLP(nn.Module):
    def __init__(self, input_size=3072, hidden_sizes=[512, 256], num_classes=10):
        super().__init__()
        self.flatten = nn.Flatten()
        
        layers = []
        prev_size = input_size
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            prev_size = hidden_size
        layers.append(nn.Linear(prev_size, num_classes))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        x = self.flatten(x)
        return self.network(x)

# 2) Инициализация модели, loss, optimizer
model = MLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 3) Функции train_one_epoch и evaluate
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return running_loss / len(train_loader), 100. * correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return running_loss / len(loader), 100. * correct / total

# 4) Логирование истории обучения
num_epochs = 10
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f'Epoch {epoch+1}/{num_epochs}: Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')

# График обучения
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train')
plt.plot(history['val_loss'], label='Val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Loss')

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train')
plt.plot(history['val_acc'], label='Val')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.title('Accuracy')
plt.tight_layout()
plt.savefig('artifacts/figures/base_training.png')
plt.show()

In [ ]:
# Пункт 3.1: Эксперименты с регуляризацией (CIFAR10)

# Используем класс MLP из пункта 2.3.3
# Функции train_one_epoch и evaluate уже реализованы в пункте 2.3.3

# Функция для обучения с логированием
def train_experiment(model, train_loader, val_loader, epochs, lr=0.001, early_stopping=None):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }
    
    best_val_acc = 0
    best_state = None
    patience_counter = 0
    
    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        print(f'Epoch {epoch+1}/{epochs}: Train Acc: {train_acc:.2f}%, Val Acc: {val_acc:.2f}%')
        
        if early_stopping:
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_state = model.state_dict().copy()
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= early_stopping:
                    model.load_state_dict(best_state)
                    print(f'Early stopping at epoch {epoch+1}')
                    break
    
    final_val_acc = best_val_acc if early_stopping else val_acc
    return history, final_val_acc

# Словарь для сохранения результатов
results = {}

# E1: Base MLP (без Dropout и BatchNorm)
print("=== E1: Base MLP ===")
model_e1 = MLP(hidden_sizes=[512, 256])
history_e1, val_acc_e1 = train_experiment(model_e1, train_loader, val_loader, epochs=15)
results['E1'] = {'val_acc': val_acc_e1, 'history': history_e1}

# E2: MLP с Dropout
print("\n=== E2: MLP with Dropout ===")
class MLPDropout(MLP):
    def __init__(self, input_size=3072, hidden_sizes=[512, 256], num_classes=10, dropout=0.3):
        super().__init__(input_size, hidden_sizes, num_classes)
        layers = []
        layers.append(nn.Flatten())
        prev_size = input_size
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_size = hidden_size
        layers.append(nn.Linear(prev_size, num_classes))
        self.network = nn.Sequential(*layers)

model_e2 = MLPDropout(hidden_sizes=[512, 256], dropout=0.3)
history_e2, val_acc_e2 = train_experiment(model_e2, train_loader, val_loader, epochs=15)
results['E2'] = {'val_acc': val_acc_e2, 'history': history_e2}

# E3: MLP с BatchNorm
print("\n=== E3: MLP with BatchNorm ===")
class MLPBatchNorm(MLP):
    def __init__(self, input_size=3072, hidden_sizes=[512, 256], num_classes=10):
        super().__init__(input_size, hidden_sizes, num_classes)
        layers = []
        layers.append(nn.Flatten())
        prev_size = input_size
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.BatchNorm1d(hidden_size))
            layers.append(nn.ReLU())
            prev_size = hidden_size
        layers.append(nn.Linear(prev_size, num_classes))
        self.network = nn.Sequential(*layers)

model_e3 = MLPBatchNorm(hidden_sizes=[512, 256])
history_e3, val_acc_e3 = train_experiment(model_e3, train_loader, val_loader, epochs=15)
results['E3'] = {'val_acc': val_acc_e3, 'history': history_e3}

# E4: Лучшая модель с EarlyStopping
print("\n=== E4: Best model with EarlyStopping ===")
best_val = max(val_acc_e2, val_acc_e3)
if best_val == val_acc_e2:
    print("Using Dropout model")
    model_e4 = MLPDropout(hidden_sizes=[512, 256], dropout=0.3)
else:
    print("Using BatchNorm model")
    model_e4 = MLPBatchNorm(hidden_sizes=[512, 256])

history_e4, val_acc_e4 = train_experiment(model_e4, train_loader, val_loader, epochs=25, early_stopping=4)
results['E4'] = {'val_acc': val_acc_e4, 'history': history_e4}

# Сохраняем лучшую модель
torch.save(model_e4.state_dict(), 'artifacts/best_model.pt')
print(f"\nBest model saved with val_acc: {val_acc_e4:.2f}%")

# График для лучшей модели (E4)
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_e4['train_loss'], label='Train')
plt.plot(history_e4['val_loss'], label='Val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Loss - Best Model (E4)')

plt.subplot(1, 2, 2)
plt.plot(history_e4['train_acc'], label='Train')
plt.plot(history_e4['val_acc'], label='Val')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.title('Accuracy - Best Model (E4)')
plt.tight_layout()
plt.savefig('artifacts/figures/curves_best.png')
plt.show()

# Финальная оценка лучшей модели на test
test_loss, test_acc = evaluate(model_e4, test_loader, criterion, device)
print(f"\nFinal test evaluation of best model: Loss: {test_loss:.4f}, Acc: {test_acc:.2f}%")

In [ ]:
# Пункт 3.2: LR, оптимизаторы, weight decay
# Используем архитектуру из E4 (лучшей модели)

# Определяем архитектуру из E4 (та же, что использовалась в лучшей модели)
class BestModel(nn.Module):
    def __init__(self, input_size=3072, hidden_sizes=[512, 256], num_classes=10, dropout=0.3, use_batchnorm=True):
        super().__init__()
        self.flatten = nn.Flatten()
        
        layers = []
        prev_size = input_size
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(hidden_size))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev_size = hidden_size
        layers.append(nn.Linear(prev_size, num_classes))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        x = self.flatten(x)
        return self.network(x)

# Функция для обучения экспериментов (короткая)
def train_short_experiment(model, train_loader, val_loader, epochs, optimizer_config):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    
    if optimizer_config['name'] == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=optimizer_config['lr'], weight_decay=optimizer_config.get('weight_decay', 0))
    elif optimizer_config['name'] == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=optimizer_config['lr'], 
                            momentum=optimizer_config.get('momentum', 0), 
                            weight_decay=optimizer_config.get('weight_decay', 0))
    
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }
    
    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        print(f'Epoch {epoch+1}/{epochs}: Train Acc: {train_acc:.2f}%, Val Acc: {val_acc:.2f}%')
    
    return history

# O1: LR слишком большой (Adam, lr=1e-1)
print("\n=== O1: LR too high (Adam, lr=0.1) ===")
model_o1 = BestModel()
history_o1 = train_short_experiment(model_o1, train_loader, val_loader, epochs=6, 
                                   optimizer_config={'name': 'Adam', 'lr': 0.1})

# O2: LR слишком маленький (Adam, lr=1e-5)
print("\n=== O2: LR too low (Adam, lr=1e-5) ===")
model_o2 = BestModel()
history_o2 = train_short_experiment(model_o2, train_loader, val_loader, epochs=6,
                                   optimizer_config={'name': 'Adam', 'lr': 0.00001})

# O3: SGD+momentum + weight decay
print("\n=== O3: SGD + momentum (0.9) + weight decay (1e-4) ===")
model_o3 = BestModel()
history_o3 = train_short_experiment(model_o3, train_loader, val_loader, epochs=12,
                                   optimizer_config={'name': 'SGD', 'lr': 0.01, 'momentum': 0.9, 'weight_decay': 0.0001})

# График для O1 и O2 (LR extremes)
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history_o1['val_acc'], label='LR=0.1 (too high)', marker='o')
plt.plot(history_o2['val_acc'], label='LR=1e-5 (too low)', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy (%)')
plt.legend()
plt.title('Effect of Extreme Learning Rates')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_o1['val_loss'], label='LR=0.1 (too high)', marker='o')
plt.plot(history_o2['val_loss'], label='LR=1e-5 (too low)', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.legend()
plt.title('Effect of Extreme Learning Rates on Loss')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('artifacts/figures/curves_lr_extremes.png')
plt.show()

# Дополнительный график для O3
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_o3['train_acc'], label='Train')
plt.plot(history_o3['val_acc'], label='Val')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.title('SGD + momentum + weight decay - Accuracy')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_o3['train_loss'], label='Train')
plt.plot(history_o3['val_loss'], label='Val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('SGD + momentum + weight decay - Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('artifacts/figures/curves_sgd.png')
plt.show()